# သင်ခန်းစာ ၁၀ - ထုတ်လုပ်မှုတွင် AI ကိုယ်စားလှယ်များ

ဤသင်ခန်းစာတွင် Microsoft Agent Framework (Python) အသုံးပြု၍ AI ကိုယ်စားလှယ်များအတွက် **ထုတ်လုပ်မှု ပုံစံများ** ကို လေ့လာသွားမည်ဖြစ်သည်။ ကျွန်ုပ်တို့ အောက်ပါအချက်များကို ဖော်ပြပါမည်။

- **မြင်သာမှုပေါ်ခြင်း** — ကိုယ်စားလှယ် မဟာဗျူဟာများတွင် အချိန်နှုန်းနှင့် မှတ်တမ်းတင်ခြင်းထည့်သွင်းခြင်း
- **သုံးသပ်ခြင်း** — ပြန်လည်တုံ့ပြန်မှု အရည်အသွေးကို ငွေရေးကြေးရေး ကိုယ်စားလှယ်မှ သတ်မှတ်ခြင်း
- **ကုန်ကျစရိတ် စီမံခန့်ခွဲမှု** — တိုကင် များအား ထိရောက်စွာ အသုံးပြုရန် နှင့် မော်ဒယ် ရွေးချယ်မှု မဟာဗျူဟာများ

အဆိုပါ ရှုထောင့်မှာ သုံးစွဲသူများအား ခရီးစဉ် စီစဉ်ရာတွင် ကူညီပေးသော **ခရီးသွား ကိုယ်စားလှယ်** ဖြစ်ပြီး၊ သတိပေးမှုနှင့် သုံးသပ်မှုများကို အထက်လွှမ်းမိုးထားသည်။


## သတ်မှတ်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ထုတ်လုပ်မှု ဆိုင်ရာ စဉ်းစားချက်များ

AI အေးဂျင့်များကို ပရိုတိုတိုက်မှ ထုတ်လုပ်မှုသို့ ရွှေ့ပေးခြင်းတွင် အဓိကအားဖြင့် ခြေသုံးချက် သုံးခုကို သတိထားရပါမည်-

1. **ကြည့်ရှုနိုင်ခြင်း** — အေးဂျင့် အလုပ်လုပ်နည်း၊ ကြာမြင့်ချိန် နှင့် အသုံးပြုသော ကိရိယာများကို မြင်နိုင်ရမည်။ ထောက်လှမ်းခြင်းနှင့် မှတ်တမ်းတင်ခြင်း မရှိပါက ထုတ်လုပ်မှုပြဿနာများကို အမှန်တကယ် ပြင်ဆင်ရန် မဖြစ်နိုင်ပါ။

2. **အကဲဖြတ်ခြင်း** — အလိုအလျောက် အရည်အသွေး စစ်ဆေးခြင်းများဖြင့် အေးဂျင့်၏ တုံ့ပြန်ချက်များ အချိန်နှင့်အမျှ တိကျမှန်ကန်၊ ပြည့်စုံပြီး အကောင်းမြန် မရှိကြောင်းကို သေချာစေသည်။ အကဲဖြတ်သူ တစ်ဦးမှ သတ်မှတ်ထားသော အခြေခံချက်များနှင့် ဆန့်ကျင်၍ တုံ့ပြန်ချက်များကို အမှတ်ပေးနိုင်ပါသည်။

3. **ကုန်ကျစရိတ် စီမံခန့်ခွဲမှု** — တိုကင်အသုံးပြုမှုသည် တိုက်ရိုက် စုစုပေါင်းကုန်ကျစရိတ်ကို ထိန်းချုပ်သည်။ ရွေးချယ်မှုဖြစ်သော prompt optimization, model selection, နှင့် caching စနစ်များက စရိတ်ကို ထိန်းသိမ်းရာတွင် အရည်အသွေးမပျက်စီးစေဘဲ အကူအညီပြုသည်။


## အာရုံပြုနိုင်သော Agent တစ်ခု တည်ဆောက်ခြင်း

ကျွန်ုပ်တို့သည် ခရီးသွားကိရိယာများကို သတ်မှတ်ပြီး agent ခေါ်ဆိုမှုကို အချိန်တိုင်းတာခြင်းဖြင့် ထုပ်ပိုးပါသည်၊ ထို့ကြောင့် နောက်ကျမှုကို ကြည့်ရှု စစ်ဆေးနိုင်သည်။ ထုတ်လုပ်မှုတွင် သင်သည် OpenTelemetry သို့မဟုတ် ဆင်တူသော တွေ့ရှိမှုပုံစံအား စနစ်တကျ ပေါင်းစည်းအသုံးပြုမည်ဖြစ်သည်။


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## သုံးသပ်မှုပုံစံများ

ထုတ်လုပ်မှုစံနှုန်းများတွင် ဒုတိယအေးဂျင့်တစ်ဦးကို **သုံးသပ်သူ**အဖြစ် အသုံးပြုသည်မှာ အများစုဖြစ်သည်။ သုံးသပ်သူသည် အဓိကအေးဂျင့်၏တုံ့ပြန်ချက်ကို ပြည့်စုံမှု၊ တိကျမှုနှင့် အသုံးဝင်မှုကဲ့သို့ ကြိုတင်သတ်မှတ်ထားသည့် စံနှုန်းများအပေါ် မူတည်၍ အမှတ်ပေးသည်။

၎င်းသည် အောက်ပါများကို ခွင့်ပြုသည်။
- အသုံးပြုသူများထံ ပြန်လည်တုံ့ပြန်မှုမရောက်မီ စက်ရုပ်အရည်အသွေးတံခါးများကို အလိုအလျောက် ထားရှိခြင်း
- တိုးတက်သွားမှုများကို စိစစ်ခြင်း၊ မေးခွန်းများ သို့မဟုတ် မော်ဒယ်များ ပြောင်းလဲသွားစဉ် တွေ့ရှိခြင်း
- အေးဂျင့်၏ တုန့်ပြန်မှုများကို အချိန်အလိုက် ဆက်လက်စောင့်ကြည့်ခြင်း


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## စွမ်းဆောင်ငွေပမာဏ စီမံခန့်ခွဲမှု မဟာဗျူဟာများ

ကုန်ကျစရိတ်များကိုထိန်းချုပ်ခြင်းသည် ထုတ်လုပ်မှု AI ကိုယ်စားလှယ်များအတွက် အလွန်အရေးကြီးသည်။ အဓိက မဟာဗျူဟာများမှာ အောက်ပါအတိုင်းဖြစ်ပါသည် –

| မဟာဗျူဟာ | ဖော်ပြချက် |
|---|---|
| **အမြန်ပြုပြင်သတ်မှတ်ခြင်း** | စနစ်ညွှန်ကြားချက်များကိုတိုတောင်းစွာ ထိန်းသိမ်းပါ။ သိပ်သည်းနေသောအချက်အလက်များကို ဖယ်ရှား၍ ထည့်သွင်းသော ကုဒ်များအရေအတွက် လျော့နည်းစေပါ။ |
    "| **မော်ဒယ် ရွေးချယ်မှု** | ရိုးရှင်းသော ကဏ္ဍများ (ဥပမာ အမျိုးအစားခွဲခြားခြင်း သို့မဟုတ် ထုတ်ယူခြင်း) အတွက် ပိုသေးငယ်သော၊ ပိုသက်သာသော မော်ဒယ်များ (ဥပမာ GPT-5-mini) ကို အသုံးပြုပါ၊ အခက်အခဲReasoningများအတွက် သာမန်ထက်ကြီးမားသော မော်ဒယ်များကို သိမ်းဆည်းထားပါ။ |\n",
| **သိုလှောင်ခြင်း** | ကိရိယာရလဒ်များနှင့် နေ့စဉ်မေးခွန်းများကို ပြန်လည်သိုလှောင်ပြီး API ခေါ်ဆိုမှုများ မထပ်ခေါ်ရစေရန်ကြိုးပမ်းပါ။ |
| **Token အကန့်အသတ်များ** | မမွန်းမသိ ခန့်မှန်းမရအောင် Response ငယ်သောအကြောင်းအရာများ ရှောင်ကြဉ်ရန် `max_tokens` ကန့်သတ်ချက်များထားပါ။ |
| **စုစည်းခေါ်ဆိုမှု** | လူသုံးသူ မေးခွန်းများစုပေါင်း၍ တစ်ခေါ် API ခေါ်ဆိုမှုတစ်ခုအဖြစ် သတ်မှတ်နိုင်သမျှလုပ်ပါ။ |

လက်တွေ့တွင်၊ အဆင့်စိုက်နည်းလမ်းသည် ကောင်းမွန်ပါသည်။ ရိုးရှင်းသော တောင်းဆိုချက်များကို မျက်နှာသာမော်ဒယ်တစ်ခုသို့ ကူးပြောင်းပေးပြီး၊ ခက်ခဲသော မေးခွန်းများကိုသာ ပိုပြီး စွမ်းဆောင်နိုင်သောထက်မြက်သော မော်ဒယ်သို့ မှီငြမ်းပါ။


## အနှစ်ချုပ်

ဒီသင်ခန်းစာမှာ သင်မှာ သင်ယူခဲ့တာတွေက:

1. **အေဂျင့်များ၏ လှုပ်ရှားမှုအပေါ်မှာ အချိန်နှင့် မှတ်တမ်းတင်ခြင်းဖြင့် မြင်သာမှု** ပေါင်းထည့်ခြင်း၊ သြဇာကြောင်းနှင့် စောင့်ကြည့်မှုအတွက် အခြေခံတည်ဆောက်ခြင်း။
2. **အေဂျင့်ဖြေကြားချက်များကို** ပြည့်စုံမှု၊ တိကျမှု၊ နှင့် အသုံးဝင်မှုတို့အရ အော်တိုမတ်အားဖြင့် သုံးသပ်ပေးသည့် အကဲဖြတ်သူ အေဂျင့်ကို အသုံးပြုခြင်း။
3. **ကုန်ကျစရိတ်များကို** prompt အကောင်းမြှင့်ခြင်း၊ မော်ဒယ်ရွေးချယ်ခြင်း၊ ကက်ရှ်သိမ်းခြင်းနှင့် token ဘတ်ဂျက်များဖြင့် စီမံခန့်ခွဲခြင်း။

ဒီထုတ်လုပ်မှုပုံစံများက သင့်ရဲ့ AI အေဂျင့်များကို ယုံကြည်စိတ်ချရမှု၊ တိုင်းတာနိုင်မှု၊ နှင့် ကျယ်ပြန့်စွာမှာ ကုန်ကျစရိတ်ထိရောက်မှုရှိစေရန် အကူအညီပြုပါသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
